# 🏨 HotelAI — تجربة سريعة للمشروع كله
**صوت ← Whisper ← RAG ← LLM (+ tools) ← رد بصوت مصري**

قبل ما تبدأ: من فوق اختار **Runtime → Change runtime type → T4 GPU**، وبعدين شغّل الخلايا بالترتيب.
كله ببلاش، ومش محتاج أي API key.

## 1) ارفع ملف المشروع
ارفع **آخر** `HotelAI_Clean.zip`. الخلية دي بتمسح أي نسخة قديمة، ولو الملف قديم هتقولك.

In [ ]:
from google.colab import files
import os, sys, zipfile, shutil, importlib
os.chdir('/content')
shutil.rmtree('/content/HotelAI_Clean', ignore_errors=True)          # امسح أي نسخة قديمة
for k in [k for k in sys.modules if (k == 'app' or k.startswith('app.'))]: del sys.modules[k]
up = files.upload()                                            # اختار HotelAI_Clean.zip (آخر نسخة)
zipfile.ZipFile(next(iter(up))).extractall('/content')
os.chdir('/content/HotelAI_Clean')
if '/content/HotelAI_Clean' not in sys.path: sys.path.insert(0, '/content/HotelAI_Clean')
importlib.invalidate_caches()
if not os.path.exists('app/prototype/agent.py'):
    raise SystemExit('⚠️ ملف الـ zip ده قديم ومافيهوش كود التجربة. نزّل آخر HotelAI_Clean.zip وارفعه تاني.')
print('تمام ✓ المشروع جاهز:', sorted(os.listdir('.')))

## 2) تسطيب المكتبات (حوالي دقيقتين)

In [ ]:
!pip -q install sentence-transformers faster-whisper edge-tts bitsandbytes accelerate
print('تمام ✓')

## 3) الإعدادات
- `LLM_NAME`: الـ 7B أحسن في المصري. لو لقيته بطيء، جرّب `Qwen/Qwen2.5-3B-Instruct`.
- `WHISPER`: لو الذاكرة مش مكفية، استخدم `medium`.

In [ ]:
LLM_NAME = 'Qwen/Qwen2.5-7B-Instruct'
WHISPER  = 'large-v3'
RAG_MODE = 'hybrid'          # bm25 + BGE-M3

## 4) تحميل الموديلات (أول مرة بتاخد 5–8 دقايق)

In [ ]:
from importlib import import_module
A = import_module('app.prototype.agent')
S = import_module('app.prototype.speech')
TA = import_module('app.prototype.test_agent')
agent = A.Agent(llm=A.QwenLLM(LLM_NAME), mode=RAG_MODE)
stt = S.STT(WHISPER)
print('الموديلات جاهزة ✓')

## 5) جرّب بالكتابة
غيّر الأسئلة زي ما تحب.

In [ ]:
print('☎️', agent.greet(), '\n')
for q in ['الفطار بكام؟', 'لو لغيت الحجز قبلها بيوم هيتخصم مني حاجة؟', 'عايز أوضة دبل من 24 لـ 27 ديسمبر لاتنين',
          'حجزي رقم B000061 اتأكد ولا لأ؟', 'فين أقرب صيدلية؟', 'التكييف في أوضتي مش شغال من امبارح!']:
    agent.greet(); r = agent.ask(q)                     # every question = a new call
    print(f"🧑 {q}\n🤖 {r['answer']}\n   tools: {[t['tool'] for t in r['tools']]} | {r['seconds']}s\n")

### مكالمة كاملة بالكتابة
الـ agent فاكر الكلام اللي قبله في نفس المكالمة.

In [ ]:
print('☎️ نور:', agent.greet())
for q in ['السلام عليكم، عايز أحجز أوضة', 'دبل من 24 لـ 27 ديسمبر لاتنين', 'طب التوين بكام؟', 'انتي روبوت؟', 'متشكر جداً']:
    r = agent.ask(q)
    print(f"🧑 {q}\n🤖 نور: {r['answer']}\n")

## 6) جرّب بصوتك 🎤 (زي مكالمة حقيقية)
الفندق هيرد الأول بالترحيب، وبعدها اتكلم **6 ثواني** (المتصفح هيطلب إذن المايك). هتسمع الرد بصوت مصري.

In [ ]:
from IPython.display import Audio, Javascript, display
from google.colab import output
from base64 import b64decode
display(Javascript('''
const sleep = t => new Promise(r => setTimeout(r, t));
const b2t = b => new Promise(r => { const f = new FileReader(); f.onloadend = e => r(e.srcElement.result); f.readAsDataURL(b); });
var record = t => new Promise(async res => {
  const s = await navigator.mediaDevices.getUserMedia({audio: true}); const rec = new MediaRecorder(s); const ch = [];
  rec.ondataavailable = e => ch.push(e.data); rec.start(); await sleep(t);
  rec.onstop = async () => { s.getTracks().forEach(x => x.stop()); res(await b2t(new Blob(ch))); }; rec.stop(); });'''))
def record(sec=6):
    print('🎙️ اتكلم دلوقتي...')
    open('input.webm', 'wb').write(b64decode(output.eval_js(f'record({sec * 1000})').split(',')[1]))
    return 'input.webm'

P = import_module('app.rag.prompt')
async def turn(sec=6):
    text, lang = stt(record(sec))
    print(f'🧑 ({lang}):', text)
    if A.needs_lookup(text):                          # زي الموظف: "ثانية واحدة يا فندم أشوفلك"
        hold = P.HOLD.get(agent.call_lang, P.HOLD['Arabic-Egyptian']); print('⏳ نور:', hold)
        display(Audio(await S.speak(hold, agent.call_lang, 'hold.mp3'), autoplay=True))
    r = agent.ask(text)
    print('🤖 نور:', r['answer'], '| tools:', [t['tool'] for t in r['tools']])
    display(Audio(await S.speak(r['answer'], r['language']), autoplay=True))

hello = agent.greet()                                   # الفندق بيرد الأول
print('☎️ نور:', hello)
display(Audio(await S.speak(hello, 'Arabic-Egyptian'), autoplay=True))
import time; time.sleep(6)                             # استنى الترحيب يخلص
await turn(6)

#### كمّل المكالمة 🎤
شغّل الخلية دي تاني وتالت عشان تكمّل الكلام في نفس المكالمة.

In [ ]:
await turn(6)

## 7) اختبار Whisper على أسئلة معروفة
اقرا السؤال اللي هيظهر بصوتك، وهيتحسب **WER**: نسبة الكلمات اللي Whisper غلط فيها. **0 يعني ممتاز**، وأقل من **0.2** كويس للمصري.

In [ ]:
import csv, random
qs = [r['question'] for r in csv.DictReader(open('evaluation/realistic_test_egyptian.csv', encoding='utf-8'))]
ref = random.choice(qs); print('📖 اقرا ده بصوتك:\n   ', ref)
hyp, _ = stt(record(7))
print('📝 Whisper سمع:', hyp, '\nWER =', S.wer(ref, hyp))

## 8) اختبار أوتوماتيك على 40 سؤال واقعي (حوالي 5 دقايق)
- **retrieval_ok:** المعلومة الصح وصلت للموديل
- **route_ok:** الموديل عمل الصح، يعني استخدم الـ tool المطلوب، أو سأل لما السؤال ناقص، أو قال الاستقبال في الطوارئ
- **facts:** نسبة الأرقام الصح في الإجابة
- كل الإجابات بتتحفظ في ملف CSV. اقرا منها بعينك، لأن التقييم الأوتوماتيك تقريبي.

In [ ]:
import pandas as pd
res, table = TA.run(agent, n=40, show=False)
display(pd.DataFrame(table))
files.download('prototype_results.csv')

## 🌍 جرّب بكذا لغة
الـ agent بيرد بلغة اللي بيكلمه. في الصوت، Whisper بيعرف اللغة لوحده، والرد بيتقري بمتكلم من نفس اللغة. جرّب الخلية اللي تحت بالكتابة، وبعدين ارجع لخلية الصوت واتكلم إنجليزي أو فرنساوي أو روسي.

In [ ]:
for q in ['الفطار بكام؟',                                  # مصري
          'ما هي سياسة إلغاء الحجز؟',                      # فصحى
          'How much is a Deluxe room per night?',          # English
          "À quelle heure est le petit-déjeuner ?",        # Français
          'Можно ли отменить бронирование бесплатно?']:    # Русский
    agent.reset(); r = agent.ask(q)
    print(f"🧑 ({r['language']}) {q}\n🤖 {r['answer']}\n   مستندات: {r['docs']} | {r['seconds']}s\n")

### اختبار أوتوماتيك على الـ 5 لغات (8 أسئلة لكل لغة، حوالي 5 دقايق)
الأسئلة جاية من جزء الـ test في الـ Gold، يعني الموديل ماشافهاش قبل كده.
- **retrieval_ok:** المعلومة الصح وصلت للموديل
- **same_language:** الرد طالع بنفس لغة السؤال
- **facts:** الأرقام الصح موجودة في الرد

In [ ]:
import pandas as pd
res_l, table_l = TA.run_languages(agent, per_lang=8, show=False)
display(pd.DataFrame(table_l))
files.download('prototype_languages.csv')

## 9) إزاي تقرا النتيجة
| | كويس | محتاج شغل |
|---|---|---|
| same_language | 1.0 تقريباً | أقل من 0.9: الموديل بيرد بلغة غلط، أكّد عليها في البرومبت |
| retrieval_ok | أعلى من 0.8 | أقل من 0.6: جرّب ترجمة الـ KB للعربي |
| route_ok | أعلى من 0.7 | أقل من كده: حسّن البرومبت أو جرّب موديل أكبر |
| facts | أعلى من 0.7 | أقل من كده: الموديل بيألف أرقام، شدّد البرومبت |
| WER | أقل من 0.2 | أعلى من 0.35: جرّب `large-v3` أو خلي الـ agent يأكد الأرقام |
| الوقت | أقل من 5 ثواني | أكتر من كده: موديل أصغر أو API |

**جرّب كمان:** غيّر `RAG_MODE` لـ `bm25` في الخلية 3 وقارن النتيجة. ولو عندكم API key (زي Groq أو OpenAI)، استخدم `A.OpenAICompatibleLLM(base_url, key, model)` بدل `QwenLLM`.